In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import matplotlib.gridspec as gridspec
import io
from PIL import Image
from tqdm import tqdm

import numpy as np
import pandas as pd

class SimulationConfig:
    def __init__(self,
                 n_patients=100,
                 median_survival_time=12,
                 max_follow_up_time=24,
                 censoring_rate=0.3,
                 survival_distribution='exponential',  # 'exponential', 'weibull', or 'piecewise'
                 weibull_shape=2,
                 # New Piecewise Settings
                 breakpoints=None, # e.g., [6, 12]
                 hazards=None,     # e.g., [0.1, 0.05, 0.02]
                 censoring_distribution='uniform',
                 seed=42):
        self.n_patients = n_patients
        self.median_survival_time = median_survival_time
        self.max_follow_up_time = max_follow_up_time
        self.censoring_rate = censoring_rate
        self.survival_distribution = survival_distribution
        self.weibull_shape = weibull_shape
        self.breakpoints = breakpoints if breakpoints is not None else []
        self.hazards = hazards if hazards is not None else []
        self.censoring_distribution = censoring_distribution
        self.seed = seed

# def generate_survival_data(config: SimulationConfig):
#     rng = np.random.default_rng(config.seed)

#     # Step 1: Generate survival times with target median
#     if config.survival_distribution == 'exponential':
#         # Exponential median = scale * ln(2) => scale = median / ln(2)
#         scale = config.median_survival_time / np.log(2)
#         survival_times = rng.exponential(scale=scale, size=config.n_patients)

#     elif config.survival_distribution == 'weibull':
#         k = float(config.weibull_shape)
#         if k <= 0:
#             raise ValueError("weibull_shape must be > 0")

#         # Weibull median = lambda * (ln 2)^(1/k) => lambda = median / (ln 2)^(1/k)
#         lam = config.median_survival_time / (np.log(2) ** (1.0 / k))

#         # NumPy weibull(a) returns samples from Weibull(k) with scale=1 (i.e., lam=1)
#         # So we multiply by lam to apply the scale
#         survival_times = lam * rng.weibull(a=k, size=config.n_patients)

#     else:
#         raise NotImplementedError("Supported: 'exponential', 'weibull'")

#     # Step 2: Determine number of censored observations
#     n_censored = int(config.censoring_rate * config.n_patients)

#     if config.max_follow_up_time is None:
#         config.max_follow_up_time = float(np.max(survival_times) * 1.1)

#     # Step 3: Generate censoring times
#     censoring_times = np.full(config.n_patients, np.inf)

#     if config.censoring_distribution == 'uniform':
#         censored_indices = rng.choice(config.n_patients, n_censored, replace=False)

#         for idx in censored_indices:
#             censoring_times[idx] = rng.uniform(
#                 0.0,
#                 min(survival_times[idx], config.max_follow_up_time)
#             )
#     else:
#         raise NotImplementedError("Only 'uniform' censoring is implemented")

#     # Step 4: Determine observed time and event indicator
#     observed_times = np.minimum(survival_times, censoring_times)
#     events = (survival_times <= censoring_times).astype(int)

#     # Step 5: Clip to max follow-up
#     observed_times = np.clip(observed_times, 0, config.max_follow_up_time)

#     return pd.DataFrame({
#         'ID': np.arange(config.n_patients),
#         'Time': observed_times,
#         'Event': events
#     })

# def generate_survival_data(config: SimulationConfig):
#     rng = np.random.default_rng(config.seed)

#     # --- Step 1: Generate survival times based on distribution ---
#     if config.survival_distribution == 'exponential':
#         scale = config.median_survival_time / np.log(2)
#         survival_times = rng.exponential(scale=scale, size=config.n_patients)

#     elif config.survival_distribution == 'weibull':
#         k = float(config.weibull_shape)
#         if k <= 0:
#             raise ValueError("weibull_shape must be > 0")
#         lam = config.median_survival_time / (np.log(2) ** (1.0 / k))
#         survival_times = lam * rng.weibull(a=k, size=config.n_patients)

#     elif config.survival_distribution == 'piecewise':
#         if not config.hazards or len(config.hazards) != len(config.breakpoints) - 1:
#             raise ValueError("hazards must have length len(breakpoints) - 1")
        
#         def generate_single_piecewise(hazards, breaks):
#             u = rng.uniform(0, 1)
#             target_h = -np.log(1 - u)  # Cumulative hazard at event time
            
#             curr_h = 0
#             curr_t = 0
#             # Define time intervals: [0, b1, b2, ..., inf]
#             times = list(breaks)
            
#             for i in range(len(hazards)):
#                 interval_len = times[i+1] - times[i]
#                 h_in_interval = hazards[i] * interval_len
                
#                 if target_h <= curr_h + h_in_interval:
#                     # Event happens in this interval
#                     return curr_t + (target_h - curr_h) / hazards[i]
                
#                 curr_h += h_in_interval
#                 curr_t = times[i+1]

#             return curr_t

#         survival_times = np.array([generate_single_piecewise(config.hazards, config.breakpoints) 
#                                    for _ in range(config.n_patients)])

#     else:
#         raise NotImplementedError("Supported: 'exponential', 'weibull', 'piecewise'")


#     n_censored = int(config.censoring_rate * config.n_patients)
#     if config.max_follow_up_time is None:
#         config.max_follow_up_time = float(np.max(survival_times) * 1.1)

#     censoring_times = np.full(config.n_patients, np.inf)
#     if config.censoring_distribution == 'uniform':
#         censored_indices = rng.choice(config.n_patients, n_censored, replace=False)
#         for idx in censored_indices:
#             censoring_times[idx] = rng.uniform(
#                 0.0, min(survival_times[idx], config.max_follow_up_time)
#             )
            
#     observed_times = np.minimum(survival_times, censoring_times)
#     events = (survival_times <= censoring_times).astype(int)
#     observed_times = np.clip(observed_times, 0, config.max_follow_up_time)

#     return pd.DataFrame({
#         'ID': np.arange(config.n_patients),
#         'Time': observed_times,
#         'Event': events
#     })


In [12]:
import numpy as np
import pandas as pd

def generate_survival_data(config):
    rng = np.random.default_rng(config.seed)

    # --- Step 1: Generate survival times based on distribution ---
    if config.survival_distribution == 'exponential':
        scale = config.median_survival_time / np.log(2)
        survival_times = rng.exponential(scale=scale, size=config.n_patients)

    elif config.survival_distribution == 'weibull':
        k = float(config.weibull_shape)
        if k <= 0:
            raise ValueError("weibull_shape must be > 0")
        lam = config.median_survival_time / (np.log(2) ** (1.0 / k))
        survival_times = lam * rng.weibull(a=k, size=config.n_patients)

    elif config.survival_distribution == 'piecewise':
        if not config.hazards or len(config.hazards) != len(config.breakpoints) - 1:
            raise ValueError("hazards must have length len(breakpoints) - 1")
        
        actual_hazards = np.array(config.hazards, dtype=float)
        actual_breakpoints = np.array(config.breakpoints, dtype=float)

        # -- ACCELERATED FAILURE TIME (AFT) SCALING --
        if getattr(config, 'median_survival_time', None) is not None:
            t_m = config.median_survival_time
            
            # 1. Find the baseline median (m_base)
            h_target = np.log(2)
            curr_h = 0.0
            m_base = None
            
            for i in range(len(actual_hazards)):
                start = actual_breakpoints[i]
                end = actual_breakpoints[i+1]
                interval_len = end - start
                
                h_in_interval = actual_hazards[i] * interval_len
                
                # If the median falls in this interval
                if curr_h + h_in_interval >= h_target:
                    m_base = start + (h_target - curr_h) / actual_hazards[i]
                    break
                
                curr_h += h_in_interval
                
            if m_base is None:
                raise ValueError("Could not find baseline median. Check if final breakpoint is inf and hazard > 0.")
                
            # 2. Calculate time scaling factor
            gamma = t_m / m_base
            
            # 3. Scale breakpoints (ignoring np.inf) and hazards
            finite_mask = np.isfinite(actual_breakpoints)
            actual_breakpoints[finite_mask] = actual_breakpoints[finite_mask] * gamma
            actual_hazards = actual_hazards / gamma

        def generate_single_piecewise(hazards, breaks):
            u = rng.uniform(0, 1)
            target_h = -np.log(1 - u)  
            
            curr_h = 0
            curr_t = 0
            times = list(breaks)
            
            for i in range(len(hazards)):
                interval_len = times[i+1] - times[i]
                h_in_interval = hazards[i] * interval_len
                
                if target_h <= curr_h + h_in_interval:
                    return curr_t + (target_h - curr_h) / hazards[i]
                
                curr_h += h_in_interval
                curr_t = times[i+1]

            return curr_t

        survival_times = np.array([generate_single_piecewise(actual_hazards, actual_breakpoints) 
                                   for _ in range(config.n_patients)])

    else:
        raise NotImplementedError("Supported: 'exponential', 'weibull', 'piecewise'")

    # --- Step 2: Apply Censoring ---
    n_censored = int(config.censoring_rate * config.n_patients)
    if getattr(config, 'max_follow_up_time', None) is None:
        config.max_follow_up_time = float(np.max(survival_times) * 1.1)

    censoring_times = np.full(config.n_patients, np.inf)
    if getattr(config, 'censoring_distribution', 'none') == 'uniform':
        censored_indices = rng.choice(config.n_patients, n_censored, replace=False)
        for idx in censored_indices:
            censoring_times[idx] = rng.uniform(
                0.0, min(survival_times[idx], config.max_follow_up_time)
            )
            
    observed_times = np.minimum(survival_times, censoring_times)
    events = (survival_times <= censoring_times).astype(int)
    observed_times = np.clip(observed_times, 0, config.max_follow_up_time)

    return pd.DataFrame({
        'ID': np.arange(config.n_patients),
        'Time': observed_times,
        'Event': events
    })

In [13]:

# 3. Kaplan-Meier Estimator
def km_estimator(df):
    df = df.sort_values('Time')
    times = [0]
    survival_prob = [1]
    prob = 1.0

    for t in sorted(df['Time'].unique()):
        d = df[(df['Time'] == t) & (df['Event'] == 1)].shape[0]
        n = df[df['Time'] >= t].shape[0]
        if n > 0:
            prob *= (1 - d / n)
            times.append(t)
            survival_prob.append(prob)

    return np.array(times), np.array(survival_prob)

# 4. Risk Table Generator
def compute_risk_table(df, intervals):
    return [np.sum(df['Time'] >= t) for t in intervals]

# 5. Visualization
def plot_km(df, time_points, survival_prob, intervals):

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.step(time_points, survival_prob, where="post", label="KM Estimate")
    ax.set_xlabel("Time (months)")
    ax.set_ylabel("Survival Probability")
    ax.set_title("Kaplan-Meier Curve with Risk Table")
    ax.grid(False)
    ax.legend()

    ax.set_xticks(intervals)
    ax.set_ylim(0, 1)
    ax.set_xlim(0, np.ceil(df["Time"].max()))
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    plt.tight_layout()
    plt.show()

# 6. Data Export
def export_data(df, filename="SimulationData/simulated_ipd.csv"):
    df.to_csv(filename, index=False)

def plot_km_with_risk_table_underneath(df, time_points, survival_prob, intervals, 
                                     arm_label="Arm A", color="#007AFF"):
    """
    Plots a KM curve with a risk table aligned underneath.
    
    Parameters:
    - color: Hex code for the line (default is Apple-style blue).
    - arm_label: Label for the risk table row.
    """
    
    # Calculate risk counts (Assuming this function exists in your scope)
    # If not, ensure this returns a list of integers matching len(intervals)
    risk_counts = compute_risk_table(df, intervals) 

    # 1. Setup Figure with tight layout logic
    fig = plt.figure(figsize=(10, 6))
    
    # hspace=0.05 brings the table up close to the x-axis of the plot
    gs = gridspec.GridSpec(2, 1, height_ratios=[5, 1], hspace=0.05) 

    # --- Top: KM Plot ---
    ax0 = plt.subplot(gs[0])
    
    # Use linewidth=2 for a bolder, modern look
    ax0.step(time_points, survival_prob, where="post", color=color, linewidth=2, label=arm_label)
    
    # Aesthetic: Minimalist Look (Remove top and right spines)
    ax0.spines['top'].set_visible(False)
    ax0.spines['right'].set_visible(False)
    
    # Grid: Add a faint vertical grid for easier reading of timepoints
    # ax0.grid(axis='x', color='#E5E5EA', linestyle='--', linewidth=0.5)
    
    ax0.set_ylabel("Survival Probability", fontsize=12, labelpad=10)
    ax0.set_ylim(0, 1)
    
    # Set X-limits explicitly to match intervals to ensure alignment
    max_time = intervals[-1] if len(intervals) > 0 else df["Time"].max()
    ax0.set_xlim(0, max_time)
    
    ax0.set_xticks(intervals)

    # --- Bottom: Risk Table ---
    ax1 = plt.subplot(gs[1], sharex=ax0)
    
    # Clean up the table area
    ax1.set_ylim(0, 1)
    ax1.axis("off") # Turn off the square box around the table
    
    # ROBUSTNESS FIX: Label Positioning
    # Instead of data coords (x=-7), use TransAxes.
    # x=-0.02 means "2% to the left of the Y-axis". It works regardless of data scale.
    ax1.text(-0.02, 0.5, arm_label, 
             transform=ax1.transAxes, 
             ha='right', va='center', 
             fontsize=12, fontweight='bold', color="#000000")

    # Place the numbers
    for i, t in enumerate(intervals):
        # Safety check to ensure we don't index out of bounds if risk_counts < intervals
        if i < len(risk_counts):
            ax1.text(t, 0.5, f"{int(risk_counts[i])}", 
                     ha='center', va='center', fontsize=11, color="#000000")
    plt.close(fig)
    return fig

def split_fig_by_pixels(fig, alpha=0.8, dpi=200, tight=False, transparent=False):
    
    if not (0 < alpha < 1):
        raise ValueError("alpha must be between 0 and 1 (exclusive).")

    # Render the figure to a PNG in memory
    buf = io.BytesIO()
    fig.savefig(
        buf,
        format="png",
        dpi=dpi,
        bbox_inches="tight" if tight else None,
        transparent=transparent,
        facecolor=fig.get_facecolor(),
        edgecolor="none",
    )
    buf.seek(0)

    # Open as PIL image (RGBA preserves transparency if requested)
    img = Image.open(buf).convert("RGBA" if transparent else "RGB")
    w, h = img.size

    # Compute split rows; make sure each part has at least 1 pixel
    split_row = max(1, min(h - 1, int(round(h * alpha))))

    # Crop: (left, top, right, bottom)
    top_img = img.crop((0, 0, w, split_row))
    bottom_img = img.crop((0, split_row, w, h))

    buf.close()
    
    return top_img, bottom_img

In [14]:
# def plot_km_only(df, time_points, survival_prob, intervals):
#     risk_counts = compute_risk_table(df, intervals)

#     fig = plt.figure(figsize=(8, 4))
#     gs = gridspec.GridSpec(2, 1, height_ratios=[4, 1])
    
#     # KM Plot
#     ax0 = plt.subplot(gs[0])
#     ax0.step(time_points, survival_prob, where="post", label="KM Estimate")
#     ax0.set_ylabel("")
#     # ax0.set_title("Kaplan-Meier Curve with Risk Table")
#     ax0.grid(False)
#     # ax0.legend(loc='upper right')
    
#     max_time = np.ceil(df["Time"].max())
#     ax0.set_xlim(0, max_time)

#     ax0.set_xticks(intervals)
#     ax0.set_ylim(0, 1.05)

#     # Risk Table
#     ax1 = plt.subplot(gs[1], sharex=ax0)
#     ax1.axis("off")
#     plt.tight_layout()
#     plt.close()
#     return fig

In [15]:
def plot_km_only(df, time_points, survival_prob, intervals):
    risk_counts = compute_risk_table(df, intervals)

    fig = plt.figure(figsize=(8, 4))
    gs = gridspec.GridSpec(2, 1, height_ratios=[4, 1])
    
    # KM Plot
    ax0 = plt.subplot(gs[0])
    ax0.step(time_points, survival_prob, where="post", label="KM Estimate")
    ax0.set_ylabel("")
    ax0.grid(False)
    
    max_time = np.ceil(df["Time"].max())
    ax0.set_xlim(0, max_time)
    ax0.set_xticks(intervals)
    ax0.set_ylim(0, 1.05)

    # Remove bounding box and reduce white space
    ax0.spines['top'].set_visible(False)
    ax0.spines['right'].set_visible(False)
    ax0.spines['left'].set_visible(True)
    ax0.spines['bottom'].set_visible(True)
    ax0.tick_params(left=True, bottom=True)

    # Risk Table
    ax1 = plt.subplot(gs[1], sharex=ax0)
    ax1.axis("off")
    plt.tight_layout(pad=0.5)  # Reduce padding to minimize white space
    plt.close()
    return fig

# Simulation Set Up

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import itertools
from uuid import uuid4

distributions = ['piecewise', 'exponential', 'weibull']

breakpoints = [3,6,9,12,15,18,21,24]
hazards = [0.01, 0.02, 0.05, 0.05, 0.1, 0.5, 0.5, 0.05, 0.01 ]

arms = ['0','1']
level_names = ['low', 'medium', 'high']

# 1. Define parameter levels
param_levels = {
    'n_patients': {
        'low': {'loc': 50,  'scale': 10},
        'medium': {'loc': 200, 'scale': 30},
        'high': {'loc': 800, 'scale': 50}
    },
    'median_survival_time': {
        'low': {'loc': 6, 'scale': 1},
        'medium': {'loc': 12, 'scale': 2},
        'high': {'loc': 36, 'scale': 6}
    },
    'censoring_rate': {
        'low': {'loc': 0.05, 'scale': 0.02},
        'medium': {'loc': 0.3, 'scale': 0.05},
        'high': {'loc': 0.7, 'scale': 0.08}
    }
}

sce_replication_num = 10

# Typical follow-up times in months, e.g. trials could be from 6 months to 5 years
def sample_followup_time():
    return np.random.choice(np.arange(6, 61, 6))  # e.g., 6, 12, ..., 60 months

# Prepare output folders
os.makedirs("../SimulationData/figures", exist_ok=True)
os.makedirs("../SimulationData/ipd", exist_ok=True)

# 2. Get all combinations of parameter levels
combinations = list(itertools.product(level_names, repeat=3))  # 3 params, 3 levels each

# 3. Collect meta data
meta_records = []

for distribution in distributions:
    for setting_idx, (n_pat_level, medsurv_level, cens_level) in enumerate(tqdm(combinations)):

        # # if (n_pat_level != 'high' or medsurv_level != 'high' or cens_level != 'low'):
        # if (n_pat_level != 'low' or medsurv_level != 'low' or cens_level != 'high'):
        #     # Skip unrealistic scenario
        #     continue

        id = 0 
        for replicate in range(sce_replication_num):
            for arm in arms:
                # Sample parameter values from normal distributions
                n_pat = int(np.clip(np.random.normal(**param_levels['n_patients'][n_pat_level]), 10, None))
                med_surv = np.clip(np.random.normal(**param_levels['median_survival_time'][medsurv_level]), 0.1, None)
                cens_rate = np.clip(np.random.normal(**param_levels['censoring_rate'][cens_level]), 0, 0.99)
                if distribution == 'piecewise':
                    max_fu = 36
                    breakpoints = np.linspace(0, max_fu, 13)
                    hazards = [0.01, 0.01, 0.01, 0.02, 0.05, 0.05, 0.1, 0.5, 0.5, 0.05, 0.01, 0.01]
                else:
                    max_fu= None
                unique_id = f"{distribution}_{n_pat_level}_{medsurv_level}_{cens_level}_{id}_{arm}"
                seed = id
                

                # Config for this replicate
                config = SimulationConfig(
                    n_patients=n_pat,
                    survival_distribution = distribution,
                    median_survival_time=med_surv,
                    max_follow_up_time=max_fu,
                    censoring_rate=cens_rate,
                    seed=seed,
                    breakpoints=breakpoints,
                    hazards=hazards
                )
                # Simulate data
                sim_data = generate_survival_data(config)
                # Compute KM
                time_pts, surv_probs = km_estimator(sim_data)
                # Plot and save
                max_time = int(np.ceil(sim_data['Time'].max()))
                intervals = np.arange(0, max_time + 1, max(1, max_time // 9))

                risk_counts = compute_risk_table(sim_data, intervals)

                plt.rcParams.update({
                    "font.size": 16,        # base font size
                    "axes.titlesize": 18,   # title font size
                    "axes.labelsize": 16,   # x and y labels
                    "xtick.labelsize": 14,  # x tick labels
                    "ytick.labelsize": 14,  # y tick labels
                    "legend.fontsize": 14   # legend text
                })

                fig = plot_km_with_risk_table_underneath(sim_data, time_pts, surv_probs, intervals)
                fig.savefig(f"../SimulationData/figures/km_{unique_id}.png", dpi = 100)
                
                _, risk_table_fig =  split_fig_by_pixels(fig, alpha=0.8)
                risk_table_fig.save(f"../SimulationData/figures/km_{unique_id}_risk_table.png")

                fig = plot_km_only(sim_data, time_pts, surv_probs, intervals)
                KM_line_fig, _ =  split_fig_by_pixels(fig, alpha=0.8)
                KM_line_fig.save(f"../SimulationData/figures/km_{unique_id}_KM_line.png")

                KM_line_fig.close()
                risk_table_fig.close()
                plt.close(fig)

                sim_data.to_csv(f"../SimulationData/ipd/simulated_ipd_{unique_id}.csv", index=False)

                # Meta record
                meta_records.append({
                    "dataset_id": unique_id,
                    "n_patients": n_pat,
                    "n_patients_level": n_pat_level,
                    "median_survival_time": med_surv,
                    "median_survival_time_level": medsurv_level,
                    "censoring_rate": cens_rate,
                    "censoring_rate_level": cens_level,
                    "max_follow_up_time": max_fu,
                    "risk_table": risk_counts
                })
            id += 1
    #         break
    #     break
    # break

# 4. Build meta data DataFrame and save
meta_df = pd.DataFrame(meta_records)
meta_df.to_csv("../SimulationData/simulation_metadata.csv", index=False)

  0%|          | 0/27 [00:00<?, ?it/s]

100%|██████████| 27/27 [04:13<00:00,  9.38s/it]
